# SteelVision — 02 · PatchCore anomaly detection on MVTec AD (`metal_nut`)

Phase 3 sanity check: train **PatchCore** (`anomalib`) on defect-free (`train/good`) images only,
evaluate **image-level AUROC** and **pixel-level AUROC** on the test set (4 defect categories + good),
export the fitted model to **ONNX**, and verify CPU inference with **ONNX Runtime**.

Runs either **locally** (CPU/GPU, ~1–2 min on CPU with `coreset_sampling_ratio=0.01`) or on **Kaggle GPU**.

In [ ]:
import sys
from pathlib import Path

# Local repo or embedded Kaggle kernel layout
for candidate in [Path("../src"), Path("src"), Path("steelvision_src")]:
    if (candidate / "steelvision").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import steelvision
print("steelvision", steelvision.__version__)

In [ ]:
from pathlib import Path
from steelvision.models.anomaly import PatchCoreConfig, train_and_evaluate

DATA_ROOT = Path("../data") if Path("../data/raw/mvtec_ad").exists() else Path("data")
OUT_DIR = Path("../runs/patchcore") if Path("../data").exists() else Path("runs/patchcore")

cfg = PatchCoreConfig(
    category="metal_nut",
    backbone="wide_resnet50_2",
    layers=("layer2", "layer3"),
    coreset_sampling_ratio=0.01,
    num_neighbors=9,
    image_size=(256, 256),
    seed=42,
)

summary = train_and_evaluate(
    cfg,
    data_root=DATA_ROOT,
    out_dir=OUT_DIR,
    export_onnx=True,
    skip_download=(DATA_ROOT / "raw" / "mvtec_ad" / "metal_nut").exists(),
)
summary

In [ ]:
# Sanity-check fusion decision rule on a good vs defective test sample
import cv2
from steelvision.decision.fusion import Detection, decide
from steelvision.models.anomaly import OnnxAnomalyDetector

detector = OnnxAnomalyDetector(summary["onnx"]["path"], input_size=(256, 256))

good_img_path = sorted((DATA_ROOT / "raw/mvtec_ad/metal_nut/test/good").glob("*.png"))[0]
ng_img_path = sorted((DATA_ROOT / "raw/mvtec_ad/metal_nut/test/scratch").glob("*.png"))[0]

for label, path in [("good", good_img_path), ("scratch", ng_img_path)]:
    rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    score, amap = detector.predict(rgb)
    verdict = decide(detections=[], anomaly_score=score, det_threshold=0.5, anom_threshold=0.5)
    print(f"{label:8s} | score={score:.4f} | verdict={verdict.verdict} (reason={verdict.reason})")